In [ ]:
import requests
import pandas as pd
import time

In [ ]:
# Replace with your actual Geoapify API key
API_KEY = ""
BASE_URL = "https://api.geoapify.com/v1/geocode/reverse"

csv_path = "CPP_Data.csv"
coordinates = pd.read_csv(csv_path).iloc[:,:3].values

# Data storage
data = []
counter = 1
# Process each coordinate
for identifier, lon, lat in coordinates:
    params = {"lat": lat, "lon": lon, "apiKey": API_KEY}
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        result = response.json()
        if result.get("features"):
            props = result["features"][0]["properties"]
            state = props.get("state", "N/A")
            county = props.get("county", "N/A")
            data.append([identifier, lon, lat, state, county])
        else:
            data.append([identifier, lon, lat, "N/A", "N/A"])
    else:
        data.append([identifier, lon, lat, "N/A", "N/A"])

    print("Reading",counter,"/ 265 has been completed.")
    counter+=1
    time.sleep(0.5)  # Delay to avoid rate limits

# Convert to DataFrame
df = pd.DataFrame(data, columns=["Identifier", "Longitude", "Latitude", "State", "County"])

# Save to CSV
df.to_csv("coordinates_with_state_county.csv", index=False)

print("Data saved to coordinates_with_state_county.csv")

In [ ]:
# Load user data
user_df = pd.read_csv('coordinates_with_state_county.csv')

# Load state FIPS codes
state_fips = pd.read_csv('state_fips.txt', delimiter=r'\s{2,}', engine='python', names=['StateFIPS', 'State'], dtype=str)
state_fips['State'] = state_fips['State'].str.upper()

# Load county FIPS codes
county_fips = pd.read_csv('county_fips.txt', delimiter=r'\s{2,}', engine='python', names=['CountyFIPS', 'County'], dtype=str)
county_fips['StateFIPS'] = county_fips['CountyFIPS'].str[:2]  # Extract state FIPS from county FIPS
county_fips['County'] = county_fips['County'].str.upper()

# Merge user data with state FIPS
user_df['State'] = user_df['State'].str.upper()
user_df = user_df.merge(state_fips, on='State', how='left')

# Merge user data with county FIPS
user_df['County'] = user_df['County'].str.upper()
user_df = user_df.merge(county_fips, on=['StateFIPS', 'County'], how='left')


# Drop 'State' and 'County' columns if they exist
user_df = user_df.drop(columns=['State', 'County'], errors='ignore')

# Reorder columns: Swap 'StateFIPS' and 'CountyFIPS'
user_df = user_df[['Identifier', 'Longitude', 'Latitude', 'CountyFIPS', 'StateFIPS']]

# Rename columns
user_df.columns = ['registryid', 'x', 'y', 'county_fips', 'state_fips']

output_file = "CPPs_with_FIPS.csv"

# Save the updated dataset
user_df.to_csv(output_file, index=False)

print(f"Processing complete. File saved as: {output_file}")

In [ ]:
# Load the datasets
cpps_fips = pd.read_csv("CPPs_with_FIPS.csv")
cpp_data = pd.read_csv("CPP_Data.csv")

# Extract the required columns
cpps_fips_selected = cpps_fips.iloc[:, :5]  # First 5 columns
cpp_data_selected = cpp_data.iloc[:, 3:]  # Columns from index 3 onwards

# Concatenate the selected columns
cpp_data_complete = pd.concat([cpps_fips_selected, cpp_data_selected], axis=1)

# Save the final dataset
cpp_data_complete.to_csv("CPP_Data_Complete.csv", index=False)

print("Processing complete. File saved as CPP_Data_Complete.csv")